In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout
from sklearn.model_selection import train_test_split

# Carga tus datos
datos = pd.read_excel("[HackMTY2025]_ConsumptionPrediction_Dataset_v1.xlsx")

In [3]:
# Eliminamos columnas que no usaremos
datos.drop('Crew_Feedback', axis=1, inplace=True)
datos.drop('Flight_ID', axis=1, inplace=True)
datos.drop('Service_Type', axis=1, inplace=True)
datos.drop('Product_ID', axis=1, inplace=True)
datos.drop('Unit_Cost', axis=1, inplace=True)

In [4]:
# Convertir la columna 'Date' a formato de fecha
datos['Date'] = pd.to_datetime(datos['Date'])

# Extraer características relevantes de la fecha
datos['Year'] = datos['Date'].dt.year
datos['Month'] = datos['Date'].dt.month
datos['DayOfWeek'] = datos['Date'].dt.dayofweek # Lunes=0, Domingo=6

# --- ¡IMPORTANTE! ---
# Guardamos los datos originales ANTES de 'get_dummies' o 'drop'
# Añadimos 'Product_Name' para usarlo en la evaluación
datos_originales_para_ver = datos[['Date', 'Origin', 'Flight_Type', 'Passenger_Count', 'Product_Name']].copy()

# Eliminamos la columna 'Date' original
datos = datos.drop('Date', axis=1) 

# --- ¡CAMBIO CLAVE! ---
# Convertimos 'Product_Name' en una característica de entrada también
datos = pd.get_dummies(datos, columns=['Origin', 'Flight_Type', 'Product_Name'])

# --- ¡CAMBIO CLAVE! ---
# Nuestra ÚNICA salida (target) es la cantidad consumida
y = datos['Quantity_Consumed']

# --- ¡CAMBIO CLAVE! ---
# 'X' es todo lo demás, eliminando targets y "trampas"
X = datos.drop(['Quantity_Consumed', 'Quantity_Returned', 'Standard_Specification_Qty'], axis=1)

In [5]:
# Ahora solo separamos X, la única y, y los datos originales
X_train, X_test, \
y_train, y_test, \
orig_train, orig_test = train_test_split(
    X, y, datos_originales_para_ver, # <-- Solo 'y'
    test_size=0.2, random_state=42
)

# --- ¡IMPORTANTE! ---
# Guardamos las columnas exactas que el modelo verá
trained_model_columns = X_train.columns.to_list()

In [6]:
# 1. Revisar si hay NaNs (Nulos)
print("--- Revisando valores nulos (NaN) ---")
print(f"NaNs en X_train: {X_train.isnull().sum().sum()}")
print(f"NaNs en y_train: {y_train.isnull().sum()}")

# 2. Limpiar los NaNs
print("\n--- Limpiando NaNs (rellenando con 0) ---")
X_train = X_train.fillna(0)
X_test = X_test.fillna(0) 

y_train = y_train.fillna(0)
y_test = y_test.fillna(0) 

# 3. Asegurar los tipos de datos (DTypes)
print("--- Forzando tipos de datos (dtypes) ---")
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')

# 'y' ahora es regresión, así que también es 'float32'
y_train = y_train.astype('float32')
y_test = y_test.astype('float32')

print("\n¡Limpieza de datos completa! Listo para entrenar.")

--- Revisando valores nulos (NaN) ---
NaNs en X_train: 0
NaNs en y_train: 0

--- Limpiando NaNs (rellenando con 0) ---
--- Forzando tipos de datos (dtypes) ---

¡Limpieza de datos completa! Listo para entrenar.


In [7]:
# --- ARQUITECTURA NUEVA (SOLO REGRESIÓN) ---

# Definimos las entradas del modelo
input_layer = Input(shape=(X_train.shape[1],), name='input_features')

# Capas ocultas (el "cerebro" del modelo)
shared_layers = Dense(128, activation='relu')(input_layer)
shared_layers = Dropout(0.2)(shared_layers) 
shared_layers = Dense(64, activation='relu')(shared_layers)

# ---- ÚNICA CABEZA DE SALIDA: Predicción de la Cantidad (Regresión) ----
# Usamos 'relu' para asegurar que el modelo no prediga cantidades negativas
quantity_output = Dense(1, activation='relu', name='quantity_output')(shared_layers)

# Unimos todo en un solo modelo
model = Model(inputs=input_layer, outputs=quantity_output)

# Vemos un resumen de la arquitectura
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_features (InputLayer)     │ (None, 23)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         3,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ quantity_output (Dense)         │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,393 (44.50 KB)

 Trainable params: 11,393 (44.50 KB)

 Non-trainable params: 0 (0.00 B)

In [25]:
# Compilamos el modelo con una sola pérdida
model.compile(optimizer='adam',
              loss='mean_squared_error', # ¡Solo una pérdida!
              metrics=['mean_absolute_error'] # (Opcional, pero útil)
             )

# Entrenamos el modelo
history = model.fit(
    X_train,
    y_train, # ¡Ya no es un diccionario!
    epochs=100,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 1402.9073 - mean_absolute_error: 29.8202 - val_loss: 1485.7517 - val_mean_absolute_error: 27.7290
Epoch 2/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 1204.5780 - mean_absolute_error: 26.2472 - val_loss: 1076.8336 - val_mean_absolute_error: 26.5929
Epoch 3/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 1167.9812 - mean_absolute_error: 26.7502 - val_loss: 890.8779 - val_mean_absolute_error: 22.1950
Epoch 4/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 1128.1096 - mean_absolute_error: 25.9533 - val_loss: 1043.1267 - val_mean_absolute_error: 22.5369
Epoch 5/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 1126.3197 - mean_absolute_error: 25.9314 - val_loss: 977.3661 - val_mean_absolute_error: 22.0646
Epoch 6/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 1138.4154 - mean_absolute_error: 25.4363 - val_loss: 927.0972 - val_mean_absolute_error: 23.4926
Epoch 7/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss

In [9]:
# Evaluar el modelo
loss, mae = model.evaluate(X_test, y_test)
print(f"\nError Absoluto Medio (MAE): {mae:.2f} unidades")
print(f"Esto significa que el modelo se equivoca, en promedio, por ~{mae:.0f} unidades.")

# --- BUCLE DE PRUEBA ACTUALIZADO ---
predicciones = model.predict(X_test)

print("\n--- Mostrando las primeras 5 predicciones detalladas ---")

for i in range(5):
    
    # 1. Obtenemos las entradas ORIGINALES
    inputs_originales = orig_test.iloc[i]
    
    # 2. Obtenemos los resultados REALES
    real_product_name = inputs_originales['Product_Name'] # Lo sacamos de 'orig_test'
    real_quantity = y_test.iloc[i]
    
    # 3. Obtenemos los resultados PREDICHOS
    predicted_quantity = predicciones[i][0]

    # 4. Imprimimos todo de forma clara
    print("=====================================================")
    print(f"PREDICCIÓN #{i+1}")
    print("=====================================================")
    
    print("--- DATOS DE ENTRADA (Originales) ---")
    print(f"  Fecha:         {inputs_originales['Date'].strftime('%Y-%m-%d')}")
    print(f"  Origen:        {inputs_originales['Origin']}")
    print(f"  Tipo de Vuelo: {inputs_originales['Flight_Type']}")
    print(f"  Pasajeros:     {inputs_originales['Passenger_Count']}")
    print(f"  Producto:      {real_product_name}") # ¡Mostramos el producto!
    
    print("\n--- RESULTADOS DEL MODELO (Cantidad) ---")
    print(f"    -> Real:    {real_quantity} unidades")
    print(f"    -> Predicha: {predicted_quantity:.2f} unidades")
    print("\n")

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1294.6604 - mean_absolute_error: 25.3778 

Error Absoluto Medio (MAE): 25.38 unidades
Esto significa que el modelo se equivoca, en promedio, por ~25 unidades.
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step

--- Mostrando las primeras 5 predicciones detalladas ---
PREDICCIÓN #1
--- DATOS DE ENTRADA (Originales) ---
  Fecha:         2025-10-06
  Origen:        LHR
  Tipo de Vuelo: short-haul
  Pasajeros:     161
  Producto:      Still Water 500ml

--- RESULTADOS DEL MODELO (Cantidad) ---
    -> Real:    53.0 unidades
    -> Predicha: 43.15 unidades


PREDICCIÓN #2
--- DATOS DE ENTRADA (Originales) ---
  Fecha:         2025-10-04
  Origen:        DOH
  Tipo de Vuelo: medium-haul
  Pasajeros:     294
  Producto:      Sparkling Water 330ml

--- RESULTADOS DEL MODELO (Cantidad) ---
    -> Real:    192.0 unidades
    -> Predicha: 93.36 unidades


PREDICCIÓN #3
--- DATOS DE ENTRADA (Originales) ---
  Fecha:         2025-10-05
  Origen:        DOH
  Tipo de V

In [10]:
def preparar_datos_para_prediccion(raw_data_dict, model_columns):
    """
    Toma un diccionario de datos crudos (humanos) y los prepara
    para que el modelo pueda hacer una predicción.
    """
    df = pd.DataFrame([raw_data_dict])

    # Procesar la Fecha
    df['Date'] = pd.to_datetime(df['Date'])
    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month
    df['DayOfWeek'] = df['Date'].dt.dayofweek
    df = df.drop('Date', axis=1)

    # Aplicar One-Hot Encoding (¡INCLUYENDO PRODUCT_NAME!)
    df = pd.get_dummies(df, columns=['Origin', 'Flight_Type', 'Product_Name'])

    # Alineación de Columnas:
    # Asegura que el df tenga EXACTAMENTE las mismas columnas que X_train
    # Rellena con 0 las columnas que no estaban en nuestros datos (ej. 'Origin_JFK')
    df = df.reindex(columns=model_columns, fill_value=0)

    df = df.astype('float32')
    
    return df

In [ ]:
# 1. Obtenemos la lista de todos los productos únicos
todos_los_productos = list(datos_originales_para_ver['Product_Name'].unique())

# 2. Definimos los datos del vuelo
datos_vuelo_base = {
    'Origin': 'DOH',
    'Date': '2025-09-26',
    'Flight_Type': 'medium-haul',
    'Passenger_Count': 272
}

# --- ¡NUEVO! DEFINE TU UMBRAL DE CONFIANZA ---
# Solo mostrará predicciones si el error (MAE) es menor que este
# porcentaje de la predicción.
# 100.0 = "Mostrar solo si el error es menor al 100% de la predicción"
# 50.0 = "Mostrar solo si el error es menor al 50% de la predicción" (Más estricto)
umbral_de_confianza_pct = 35.0 
# epoca 50 = 30 umbral
# epoca 100 = 35 umbral

print("\n\n====================================================")
print(f"  GENERANDO LISTA DE COMPRAS PARA VUELO (OBJETO)  ")
print(f"  Origen: {datos_vuelo_base['Origin']}, Pasajeros: {datos_vuelo_base['Passenger_Count']}")
print(f"  (Umbral de confianza: Error < {umbral_de_confianza_pct}%)")
print("====================================================")

# 3. Este será tu "objeto" de predicciones
objeto_de_predicciones = {}

# 4. Iteramos sobre cada producto
for producto in todos_los_productos:
    
    # 5. Creamos la entrada completa
    datos_completos = datos_vuelo_base.copy()
    datos_completos['Product_Name'] = producto
    
    # 6. Preparamos los datos
    datos_listos = preparar_datos_para_prediccion(
        datos_completos, 
        trained_model_columns
    )
    
    # 7. Hacemos la predicción
    prediccion_cantidad = model.predict(datos_listos, verbose=0)[0][0]
    
    # 8. Decidimos si añadirlo (¡FILTRO MEJORADO!)
    if prediccion_cantidad > 1.0: # Filtro 1: Ignorar si es casi cero
        
        # --- ¡NUEVO! Calculamos el error relativo ---
        # (Asegúrate de que 'mae' exista de tu celda de 'model.evaluate')
        error_relativo_pct = (mae / prediccion_cantidad) * 100
        
        # --- ¡NUEVO! Filtro 2: Comparamos con tu umbral ---
        if error_relativo_pct < umbral_de_confianza_pct:
            # Guardamos un diccionario con más datos
            objeto_de_predicciones[producto] = {
                'cantidad': int(round(prediccion_cantidad)),
                'error_pct': error_relativo_pct
            }

# 9. ¡Imprimimos el objeto final!
print("\n--- RECOMENDACIÓN FINAL DEL MODELO ---")
print(f"(Margen de error promedio: +/- {mae:.0f} unidades por producto)")

# Ordenamos (¡modificado para el nuevo diccionario!)
predicciones_ordenadas = sorted(
    objeto_de_predicciones.items(), 
    key=lambda item: item[1]['cantidad'], 
    reverse=True
)

if not predicciones_ordenadas:
    print("El modelo no predice un consumo significativo o confiable para este vuelo.")
else:
    for producto, datos_pred in predicciones_ordenadas:
        
        cantidad = datos_pred['cantidad']
        error_pct = datos_pred['error_pct']
        
        rango_min = int(cantidad - mae)
        if rango_min < 0: rango_min = 0 
        rango_max = int(cantidad + mae)
        
        # Mostramos el error relativo de ESTA predicción
        print(f"  - {producto}: {cantidad} unidades (Rango: {rango_min}-{rango_max}) (Error Relativo: {error_pct:.0f}%)")



  GENERANDO LISTA DE COMPRAS PARA VUELO (OBJETO)  
  Origen: DOH, Pasajeros: 272
  (Umbral de confianza: Error < 35.0%)

--- RECOMENDACIÓN FINAL DEL MODELO ---
(Margen de error promedio: +/- 25 unidades por producto)
  - Sparkling Water 330ml: 83 unidades (Rango: 57-108) (Error Relativo: 30%)
  - Still Water 500ml: 83 unidades (Rango: 57-108) (Error Relativo: 30%)
  - Juice 200ml: 83 unidades (Rango: 57-108) (Error Relativo: 30%)
  - Snack Box Economy: 75 unidades (Rango: 49-100) (Error Relativo: 34%)
  - Bread Roll Pack: 74 unidades (Rango: 48-99) (Error Relativo: 34%)


In [34]:
# guardar el modelo

import joblib

# 1. Guarda el modelo
model.save('consumption_model.keras') # El nuevo formato preferido

# 2. Guarda las columnas
joblib.dump(trained_model_columns, 'model_columns.pkl')

# 3. Guarda la lista de productos
todos_los_productos = list(datos_originales_para_ver['Product_Name'].unique())
joblib.dump(todos_los_productos, 'all_products.pkl')

print("¡Modelo y activos guardados!")

¡Modelo y activos guardados!
